# 01. Senoidales: frecuencia, periodo, velocidad angular y muestreo

**Objetivos de aprendizaje.** Al terminar podrás:

1. Explicar y calcular $F$, $T$, $\Omega$ de una senoidal **analógica**.
2. Explicar y calcular $f_s$, $T_s$ y el instante $t_n$ de cada muestra.
3. Explicar y calcular la **frecuencia digital** $f$ y $\omega$ de una senoidal **digital**, y sus unidades.
4. Generar una senoidal de dos formas: **muestreando** $x(t)$ o usando **directamente** $f$.
5. Decidir si una senoidal digital es **periódica** y calcular su periodo $N$.
6. Entender por qué en digital **sólo importan las frecuencias entre 0 y 0.5** ciclos/muestra (y qué es el *aliasing*).

**Requisito:** notebook 00.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## 1. Repaso: la senoidal analógica

$$x(t) = A\cos(\Omega t + \phi) = A\cos(2\pi F t + \phi)$$

| Símbolo | Nombre | Unidades | Relación |
|---|---|---|---|
| $A$ | amplitud | (V, Pa, mV...) | valor máximo |
| $F$ | frecuencia | Hz = ciclos/s | cuántos ciclos completa en 1 segundo |
| $T$ | periodo | s | $T = 1/F$: cuánto dura un ciclo |
| $\Omega$ | velocidad (frecuencia) angular | rad/s | $\Omega = 2\pi F$: cuántos radianes avanza el ángulo en 1 s |
| $\phi$ | fase | rad | ángulo en $t = 0$ |

**¿Por qué $2\pi$?** Un ciclo completo es una vuelta de $2\pi$ radianes. Si hay $F$ ciclos por segundo,
el ángulo avanza $2\pi F$ radianes por segundo.

### Ejemplo: la red eléctrica (60 Hz). Paso a paso:

In [ ]:
# Paso 1: datos
A = 170          # amplitud en volts (≈ 120 V RMS · √2)
F = 60           # frecuencia en Hz

# Paso 2: periodo
T = 1 / F
print(f"Periodo T = 1/F = {T:.5f} s = {T*1000:.3f} ms")

# Paso 3: velocidad angular
Omega = 2 * np.pi * F
print(f"Velocidad angular Ω = 2πF = {Omega:.2f} rad/s")

Las mismas cuentas con las funciones de la biblioteca (`funciones_pds/conceptos.py`):

In [ ]:
print("T =", pds.periodo_analogico(60), "s")
print("Ω =", pds.velocidad_angular(60), "rad/s")

Dibujemos 3 ciclos. En la computadora **nada es realmente continuo**: para que una señal *parezca* analógica,
la muestreamos con una $f_s$ **muchísimo** mayor que su frecuencia (aquí, 2000 puntos en toda la duración) y
la dibujamos con línea. En este curso, cada vez que veas `fs_grafica` es eso: una señal digital tan densa que la tratamos como analógica.

In [ ]:
fs_grafica = 2000 / (3 * T)       # 2000 puntos en 3 periodos: suficiente para que se vea continua
_, t, x_t = pds.senoidal_digital(F=60, fs=fs_grafica, duracion=3*T, A=170)
pds.graficar_continua(t, x_t, titulo="x(t) = 170 cos(2π·60·t)", etiqueta_y="volts")
plt.axvline(T, color="r", linestyle="--", label=f"T = {T*1000:.2f} ms")
plt.legend()
plt.show()

## 2. Muestreo: de $x(t)$ a $x[n]$

Muestrear es **medir** la señal cada $T_s$ segundos:

$$x[n] = x(t_n), \qquad t_n = n\,T_s = \frac{n}{f_s}, \qquad n = 0, 1, 2, \dots$$

| Símbolo | Nombre | Unidades | Relación |
|---|---|---|---|
| $f_s$ | frecuencia de muestreo | muestras/s (Hz) | cuántas muestras tomamos por segundo |
| $T_s$ | periodo de muestreo | s | $T_s = 1/f_s$: tiempo entre dos muestras |
| $n$ | índice de la muestra | (sin unidades, entero) | número de muestra |
| $t_n$ | instante de la muestra $n$ | s | $t_n = n\,T_s$ |

### Ejemplo paso a paso: senoidal de $F = 5$ Hz muestreada a $f_s = 50$ Hz durante 0.4 s

In [ ]:
# Paso 1: parámetros
F = 5            # Hz
fs = 50          # muestras por segundo
duracion = 0.4   # segundos

# Paso 2: periodo de muestreo
Ts = 1 / fs
print(f"Ts = 1/fs = {Ts} s  (tomamos una muestra cada {Ts*1000:.0f} ms)")

# Paso 3: ¿cuántas muestras? N = duración. fs
N = int(duracion * fs)
print(f"N = duración · fs = {N} muestras")

In [ ]:
# Paso 4: índices de las muestras
n = np.arange(0, N)
print("n =", n)

In [ ]:
# Paso 5: instante de cada muestra  t_n = n. Ts
t_n = n * Ts
print("t_n =", t_n)

In [ ]:
# Paso 6: evaluamos la senoidal en esos instantes
x_n = np.cos(2 * np.pi * F * t_n)
print("x[n] =", np.round(x_n, 3))

In [ ]:
# Paso 7: dibujamos la señal analógica y sus muestras
_, t_c, x_c = pds.senoidal_digital(F=5, fs=2000 / duracion, duracion=duracion)   # "analógica": fs enorme
pds.graficar_muestreo(t_c, x_c, t_n, x_n, titulo="F = 5 Hz muestreada a fs = 50 Hz")
plt.show()

Observa: en un periodo ($T = 0.2$ s) caben $T/T_s = f_s/F = 10$ muestras.

La función `pds.senoidal_digital` hace los pasos 1-6 por nosotros:

In [ ]:
n, t_n, x_n = pds.senoidal_digital(F=5, fs=50, duracion=0.4)
print("primeras 5 muestras:", np.round(x_n[:5], 3))

## 3. La frecuencia DIGITAL

Sustituyamos $t_n = n/f_s$ en la senoidal:

$$x[n] = A\cos\left(2\pi F \frac{n}{f_s} + \phi\right) = A\cos\left(2\pi \underbrace{\frac{F}{f_s}}_{f}\, n + \phi\right) = A\cos(\omega n + \phi)$$

Los segundos **desaparecieron**: la señal digital sólo depende de la razón $f = F/f_s$.

| Símbolo | Nombre | Unidades | Relación | Interpretación |
|---|---|---|---|---|
| $f$ | frecuencia digital | **ciclos/muestra** | $f = F/f_s$ | qué fracción de ciclo avanza entre una muestra y la siguiente |
| $\omega$ | frecuencia angular digital | **rad/muestra** | $\omega = 2\pi f = \Omega T_s$ | cuántos radianes avanza entre muestras |
| $1/f$ | muestras por ciclo | muestras | $1/f = f_s / F$ | |

> Un detalle importante: la computadora **no sabe** cuántos Hz tiene una señal: sólo conoce las muestras. Para ella, una senoidal
> de 5 Hz muestreada a 50 Hz y una de 500 Hz muestreada a 5000 Hz son **exactamente la misma señal** ($f = 0.1$).

In [ ]:
F, fs = 5, 50
f = pds.frecuencia_digital(F, fs)
w = pds.frecuencia_angular_digital(F, fs)
print(f"f = F/fs = {f} ciclos/muestra   ->  {1/f:.0f} muestras por ciclo")
print(f"ω = 2πf  = {w:.4f} rad/muestra  =  {w/np.pi:.2f}·π rad/muestra")

In [ ]:
# Comprobación de la afirmación del recuadro: 5 Hz @ 50 Hz  vs  500 Hz @ 5000 Hz
_, _, x_a = pds.senoidal_digital(F=5, fs=50, duracion=0.4)
_, _, x_b = pds.senoidal_digital(F=500, fs=5000, duracion=0.004)
pds.comprobar_iguales(x_a, x_b, "las dos señales digitales son idénticas")

### Forma 2: generar la senoidal directamente con $f$

Si ya conocemos $f$, no necesitamos $F$ ni $f_s$:
$$x[n] = A\cos(2\pi f n + \phi), \qquad n = 0, 1, \dots, N-1$$

In [ ]:
n, x_digital = pds.senoidal_digital_muestras(N=20, f=0.1)
pds.graficar_discreta(n, x_digital, titulo="x[n] = cos(2π·0.1·n): 10 muestras por ciclo")
plt.show()

pds.comprobar_iguales(x_digital, x_a, "las dos formas (muestreo y frecuencia digital) dan lo mismo")

Para ver el código de la función (y copiarlo a tus propios scripts):

In [ ]:
import inspect
print(inspect.getsource(pds.senoidal_digital_muestras))

## 4. ¿En qué instante se tomó cada muestra?

$$t_n = \frac{n}{f_s} \qquad\Longleftrightarrow\qquad n = t \cdot f_s \;(\text{redondeado})$$

Ejemplo: un audio de calidad CD tiene $f_s = 44\,100$ Hz.

In [ ]:
fs = 44100
print("La muestra 22050 se tomó en t =", pds.tiempo_de_muestra(22050, fs), "s")
print("En t = 1.5 s está la muestra n =", pds.indice_de_muestra(1.5, fs))
print("Tiempo entre muestras Ts =", pds.periodo_de_muestreo(fs) * 1e6, "µs")

> **Error frecuente:** si una señal tiene $N$ muestras, su **duración** es $N/f_s$, pero la **última muestra**
> se tomó en $t = (N-1)/f_s$, porque la primera se tomó en $t = 0$.

In [ ]:
N = 44100                 # un segundo de audio
print("Duración          =", N / fs, "s")
print("Última muestra en =", pds.tiempo_de_muestra(N - 1, fs), "s")

## 5. ¿Es periódica una senoidal digital?

Una señal digital es periódica con periodo $N$ (entero) si $x[n + N] = x[n]$ para toda $n$.
Para la senoidal:
$$\cos(2\pi f (n+N)) = \cos(2\pi f n + 2\pi f N) \;\Rightarrow\; \text{necesitamos } fN = k \text{ (entero)}$$

> **Una senoidal digital es periódica sólo si $f$ es un número racional $f = k/N$.**
> Con la fracción irreducible, el periodo fundamental es $N$ y en ese periodo caben $k$ ciclos.

Esto no pasa en analógico, donde **toda** senoidal es periódica.

In [ ]:
for f in [1/8, 0.3, 3/7, 1/(2*np.pi)]:
    N = pds.periodo_fundamental_digital(f)
    print(f"f = {f:.5f}  ->  periodo N = {N}")

In [ ]:
figura, ejes = plt.subplots(3, 1, figsize=(10, 7))

n, x = pds.senoidal_digital_muestras(N=40, f=1/8)
pds.graficar_discreta(n, x, titulo="f = 1/8: periódica, N = 8 (1 ciclo cada 8 muestras)", ax=ejes[0])

n, x = pds.senoidal_digital_muestras(N=40, f=0.3)
pds.graficar_discreta(n, x, titulo="f = 3/10: periódica, N = 10 (3 ciclos cada 10 muestras)", ax=ejes[1])

n, x = pds.senoidal_digital_muestras(N=40, f=1/(2*np.pi))
pds.graficar_discreta(n, x, titulo="f = 1/(2π), es decir ω = 1 rad/muestra: NO periódica", ax=ejes[2])

plt.tight_layout()
plt.show()

## 6. Las frecuencias digitales se "repiten": sólo importa $0 \le f \le 0.5$

**Propiedad 1.** Sumar un número entero a $f$ no cambia nada, porque $\cos(2\pi(f+1)n) = \cos(2\pi f n + 2\pi n) = \cos(2\pi f n)$.
Equivalente: $\omega$ y $\omega + 2\pi$ son la **misma** frecuencia.

In [ ]:
n, x1 = pds.senoidal_digital_muestras(N=20, f=0.1)
n, x2 = pds.senoidal_digital_muestras(N=20, f=1.1)
pds.comprobar_iguales(x1, x2, "f = 0.1 y f = 1.1 producen exactamente las mismas muestras")

**Propiedad 2.** $f$ y $1-f$ también dan las mismas muestras (porque el coseno es par).
Entonces, la oscilación **más rápida posible** es $f = 0.5$ ($\omega = \pi$): $x[n] = \cos(\pi n) = (-1)^n$, que cambia de signo en cada muestra.

In [ ]:
figura, ejes = plt.subplots(1, 3, figsize=(13, 3))
for eje, f in zip(ejes, [0.45, 0.5, 0.55]):
    n, x = pds.senoidal_digital_muestras(N=20, f=f)
    pds.graficar_discreta(n, x, titulo=f"f = {f}", ax=eje)
plt.tight_layout()
plt.show()

Observa que $f = 0.55$ se ve **igual** que $f = 0.45$: más allá de 0.5, la frecuencia "aparente" vuelve a bajar.

## 7. Consecuencia: el teorema de muestreo (Nyquist)

Como $f = F/f_s$ debe quedar entre 0 y 0.5 para no confundirse con otra frecuencia:

$$\frac{F}{f_s} \le \frac{1}{2} \quad\Longleftrightarrow\quad f_s \ge 2F$$

Si no se cumple, la senoidal "se disfraza" de otra frecuencia más baja: **aliasing**.

In [ ]:
pds.resumen_senoidal(F=1000, fs=8000)
pds.resumen_senoidal(F=7000, fs=8000)

In [ ]:
# Comprobación: 7000 Hz muestreada a 8000 Hz da las MISMAS muestras que 1000 Hz
fs = 8000
duracion = 0.002
fs_grafica = 4000 / duracion                                      # "analógicas": fs enorme
_, t_c, x_7k = pds.senoidal_digital(7000, fs_grafica, duracion)
_, t_c, x_1k = pds.senoidal_digital(1000, fs_grafica, duracion)
n, t_n, x_n = pds.senoidal_digital(F=7000, fs=fs, duracion=duracion)

plt.figure(figsize=(10, 3))
plt.plot(t_c, x_7k, color="0.75", label="7000 Hz (original)")
plt.plot(t_c, x_1k, "g--", label="1000 Hz (alias)")
plt.stem(t_n, x_n, linefmt="C3-", markerfmt="C3o", basefmt="k-", label="muestras a 8000 Hz")
plt.legend(loc="upper right")
plt.xlabel("t (s)")
plt.title("Aliasing: las muestras no pueden distinguir 7000 Hz de 1000 Hz")
plt.show()

## 8. Escuchar una senoidal

En el notebook 03 veremos audio con detalle. Por ahora: un "La" de 440 Hz durante 1 segundo.

In [ ]:
fs = 8000
n, t, x = pds.senoidal_digital(F=440, fs=fs, duracion=1.0)
pds.reproducir(x, fs)

## Resumen

| Analógica | Muestreo | Digital |
|---|---|---|
| $x(t) = A\cos(2\pi F t+\phi)$ | $t_n = n T_s = n/f_s$ | $x[n] = A\cos(2\pi f n+\phi)$ |
| $F$ [Hz], $T = 1/F$ [s] | $f_s$ [Hz], $T_s = 1/f_s$ [s] | $f = F/f_s$ [ciclos/muestra] |
| $\Omega = 2\pi F$ [rad/s] | | $\omega = 2\pi f = \Omega T_s$ [rad/muestra] |
| siempre periódica | | periódica sólo si $f = k/N$ racional |
| todas las frecuencias son distintas | | sólo $0 \le f \le 0.5$ son distintas → $f_s \ge 2F$ |

## Comprueba tu comprensión

1. Una senoidal de 100 Hz se muestrea a 1 kHz. ¿Cuántas muestras por ciclo hay? ¿Cuánto vale $\omega$?
2. ¿$x[n] = \cos(0.5 n)$ es periódica?
3. Un audio tiene $f_s = 16$ kHz. ¿En qué instante está la muestra 4000?
4. ¿Se puede representar sin aliasing un tono de 5 kHz con $f_s = 8$ kHz? ¿Cómo sonaría?

<details><summary>Ver respuestas</summary>

1. $f_s/F = 10$ muestras por ciclo; $f = 0.1$, $\omega = 0.2\pi$ rad/muestra.
2. No: $\omega = 0.5 \Rightarrow f = 0.5/(2\pi) = 1/(4\pi)$, irracional.
3. $t = 4000/16000 = 0.25$ s.
4. No ($5 > 4$ kHz). Sonaría como $8 - 5 = 3$ kHz.
</details>